# BLNetwork Tutorial: Example 1 (Boston Housing)

This notebook is a practical tutorial for **continuous-task** workflows in blnetwork.

### 1. Load And Preprocess Boston Housing

In [1]:
import torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

In [2]:
data = fetch_openml(data_id=531, as_frame=True) 
df = data.frame
df = df.drop(columns=['PTRATIO'])  

target_col = "MEDV"

X_raw = df.drop(columns=[target_col])
y_raw = df[[target_col]]

categorical_cols = X_raw.select_dtypes(include=["category", "object"]).columns.tolist()

if categorical_cols:
    enc = OrdinalEncoder()
    X_raw[categorical_cols] = enc.fit_transform(X_raw[categorical_cols])

X_np = X_raw.to_numpy()
y_np = y_raw.to_numpy()

scaler_x = StandardScaler()
scaler_y = StandardScaler()

x_scaled = torch.tensor(scaler_x.fit_transform(X_np), dtype=torch.float32)
y_scaled = torch.tensor(scaler_y.fit_transform(y_np), dtype=torch.float32)


x_train, x_temp, y_train, y_temp = train_test_split(x_scaled, y_scaled, test_size=0.4, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

### Data Notes

- x_train, x_val, x_test should all be tensors
- For this notebook pipeline, keep target shape as (N, 1), however, BLNetwork continuous models can also accept 1D (N) targets.


## 2. Train A BLDeep Score Model

Core components used below:
- BLDeep(...): BL model (network structure and score function parameterization)
- OptimConfig(...): optimization hyperparameters
- TrainConfig(...): training hyperparameters
- ContinuousTrainer(...): applies DSM-based training for continuous targets
- ExportConfig(...): for exporting readable structure files.

In [3]:
from __future__ import annotations
import pandas as pd

from blnetwork.model import BLDeep
from blnetwork.training import OptimConfig, TrainConfig, ContinuousTrainer, ExportConfig

seed = 42
device = "cpu"

column_names = X_raw.columns.tolist()
print(f"Feature names: {column_names}")
print(f"Num features: {len(column_names)}")

input_names = column_names + [target_col]
feature_df = pd.DataFrame(columns=input_names)

ModelHyperparameters = {
    "hidden_dims": [2, 1],           # BL depth 
    "first_act_func": "tanh",        # or "none", default is "tanh"
    "second_act_func": "softplus",   # or "relu", default is "relu"
    "third_act_func": "abs",         # or "square", default is "abs"
    "head_bias": True,               # whether to include bias term in the head layer, default is True
    "task": "continuous",            # or "discrete"
    "constrain_lambda": True,        # positive lambda constraint, default is True
    "init_lambda": 1.0,              # default is 1.0
    "beta": 1.0                      # default is 1.0, only used when second_act_func is "softplus"
}

model = BLDeep(**ModelHyperparameters)

optim_cfg = OptimConfig(
    optim="adam", 
    lr=5e-3, 
    weight_decay=1e-4
)

train_cfg = TrainConfig(
    max_epochs=200,
    batch_size=64,
    early_stop=True,
    patience=15,
    device=device,
    log_every=10,
    seed=seed,
    verbose=True
)

export_cfg = ExportConfig(
    enabled=True,                               # export the model coefficients to a txt file after training
    model_path="boston_housing_model_structure.txt",      # path to save the exported txt file
    df=feature_df,                              # use df column names as txt column names
    ndigits=2,                                  # keep 2 digits for the coefficients
    tol=0.0                                     # only export terms with abs(coef) >= tol
)

trainer = ContinuousTrainer(
    model=model,
    optim_cfg=optim_cfg,
    train_cfg=train_cfg,
    dsm_sigma=1.2,
    n_noise=1,
    temperature=1.0,
    export_cfg=export_cfg
)

result = trainer.fit(x_train, y_train, x_val, y_val)


Feature names: ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'B', 'LSTAT']
Num features: 12
[Epoch 0010] train_loss=0.353180 val_loss=0.278678
[Epoch 0020] train_loss=0.194578 val_loss=0.131996
[Epoch 0030] train_loss=0.095918 val_loss=0.142014
Early stopping at epoch 39, best_epoch=24, best_val_loss=0.074096


### Inference

In [4]:
import torch
from blnetwork.training import fit_amortized_predictor, AmortizedConfig
from blnetwork.inference import predict_continuous
import torch.nn as nn


amortized_cfg = AmortizedConfig(
    epochs=300,
    weight_decay=1e-4,
    seed=seed,
    device=device
)

# A simple MLP regressor for amortized prediction
class MLPRegressor(nn.Module):
    def __init__(self, x_dim: int, y_dim: int, hidden=(128,)):
        super().__init__()
        layers = []
        d = x_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            d = h
        layers += [nn.Linear(d, y_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

predictor = MLPRegressor(x_dim=x_train.shape[1], y_dim=1)

amortized_result = fit_amortized_predictor(
    predictor=predictor,                      
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    bl_model=model,
    cfg=amortized_cfg,
)

y_hat_test = predict_continuous(predictor, x_test, device=device, return_cpu=True)

# compute some common regression metrics
mse_test = torch.mean((y_hat_test - y_test) ** 2).item()
ss_res = torch.sum((y_hat_test - y_test) ** 2).item()
ss_tot = torch.sum((y_test - torch.mean(y_test)) ** 2).item()
r2_test = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

print(f"[amortized] test_mse={mse_test:.6f} test_r2={r2_test:.4f}")


[amortized] test_mse=0.172790 test_r2=0.8301


### Some import Training Hyperparameters for Training BL

BLDeep

- hidden_dims: BL depth 
- second_act_func: C-block activation function
- third_act_func: T-block activation function
- head_bias: output layer bias term
- num_classes: number of classes K; required only when you wanna inference without training before. 
- task: task type (continuous / discrete)
- constrain_lambda: lambda positivity constraint
- beta: C-block softplus temperature

OptimConfig:
- optim: optimizer (adam, adamw, sgd)
- lr: learning rate
- weight_decay
- momentum: only required for SGD

TrainConfig:
- max_epochs
- batch_size
- num_workers: data loading parallelism
- pin_memory: CUDA memory acceleration flag
- grad_clip_norm: gradient clipping threshold
- mixed_precision: AMP training mode
- early_stop: bool (True/False)
- patience
- min_delta
- mode
- log_every: logging frequency 
- verbose: training output verbosity
- device: computation device


ContinuousTrainer:
- optim_cfg
- train_cfg
- dsm_sigma, 
- n_noise, 	
- temperature
- export_cfg

